In [ ]:
# fr/python-101/hard/06-normalizing-bigrams
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("slm-corpus.csv", ())


Des comptes aux probabilités

Les comptes bruts vous disent que « the » → « cat » est apparu 15 fois et « the » → « dog » 5 fois. Mais pour **échantillonner** le mot suivant, vous avez besoin de probabilités : « cat » devrait être choisi 75 % du temps et « dog » 25 %. La normalisation convertit les comptes en une distribution où tous les mots suivants somment à 1,0.

## Concepts clés

### Normaliser avec une boucle

Pour chaque mot, additionnez ses comptes de mots suivants, puis divisez chaque compte par ce total :


In [ ]:
def normalize_bigrams(bigrams):
    normalized = {}
    for word, followers in bigrams.items():
        total = sum(followers.values())
        normalized[word] = {w: c / total for w, c in followers.items()}
    return normalized


Maintenant, `normalized["the"]["cat"]` renvoie un float entre 0 et 1 — la probabilité que « cat » suive « the ».

### Exemple


In [ ]:
raw_bigrams = {"the": {"cat": 15, "dog": 5, "bird": 10}}
norm = normalize_bigrams(raw_bigrams)

print(norm["the"])
# {'cat': 0.5, 'dog': 0.1667, 'bird': 0.3333}


Les probabilités somment à 1,0 :


In [ ]:
print(sum(norm["the"].values()))  # 1.0


### Pourquoi la normalisation compte pour l'échantillonnage

`random.choices()` a besoin de poids qui représentent une vraisemblance relative. Si vous passez des comptes bruts (15, 5, 10), cela fonctionne — mais avoir de vraies probabilités (0,5, 0,167, 0,333) rend le modèle portable et comparable entre différentes tailles de corpus.


In [ ]:
import random

followers = list(norm["the"].keys())
weights = list(norm["the"].values())
next_word = random.choices(followers, weights=weights, k=1)[0]
print(f"Next word: {next_word}")


### Gérer les cas limites

Certains mots n'ont pas de mots suivants (le dernier mot du corpus, ou les mots qui n'apparaissent qu'à la fin d'une phrase). La table de bigrammes n'aura pas d'entrées pour eux :


In [ ]:
def normalize_bigrams(bigrams):
    normalized = {}
    for word, followers in bigrams.items():
        if not followers:
            continue  # skip words with no followers
        total = sum(followers.values())
        normalized[word] = {w: c / total for w, c in followers.items()}
    return normalized


Sauter les entrées vides évite les erreurs de division par zéro.

### Un pipeline complet

Voici comment la normalisation s'insère dans le pipeline complet :


In [ ]:
texts = load_corpus("slm-corpus.csv")
tokens = tokenize(" ".join(texts))
bigrams = build_bigrams(tokens)
model = normalize_bigrams(bigrams)

# Check a sample
print(f"Words in model: {len(model)}")
print(f"Followers of 'the': {list(model.get('the', {}).keys())[:5]}")


### Sauvegarder le modèle

Vous voudrez peut-être sauvegarder la table de bigrammes normalisée pour la réutiliser. Comme c'est un dict imbriqué de floats, `json` fonctionne bien :


In [ ]:
import json

with open("bigram_model.json", "w") as f:
    json.dump(model, f)

# Reload later
with open("bigram_model.json") as f:
    model = json.load(f)


## Essayez

Construisez et normalisez la table de bigrammes, puis vérifiez :
1. Les probabilités pour « the » somment-elles à 1,0 ?
2. Combien de mots ont zéro mot suivant ?
3. Quel est le mot le plus probable qui suit « the » ?


In [ ]:
model = normalize_bigrams(bigrams)
the_followers = model.get("the", {})
top_follower = max(the_followers, key=the_followers.get)
print(f"Most likely after 'the': '{top_follower}' ({the_followers[top_follower]:.3f})")


## Points clés

- La normalisation convertit les comptes bruts en probabilités qui somment à 1,0 par mot
- `random.choices()` utilise ces probabilités comme poids pour l'échantillonnage pondéré
- Sautez les mots sans mots suivants pour éviter la division par zéro
- Sauvegardez les modèles normalisés avec `json.dump()` pour les réutiliser entre scripts

## Défi pratique

Écrivez une fonction `bigram_stats(model)` qui affiche pour chaque mot : le mot, le nombre de mots suivants et le mot suivant le plus probable. Limitez la sortie aux 10 premiers mots par nombre total de mots suivants.


In [ ]:
def bigram_stats(model, top_n=10):
    words = sorted(model, key=lambda w: sum(model[w].values()), reverse=True)
    for word in words[:top_n]:
        followers = model[word]
        total = sum(followers.values())
        best = max(followers, key=followers.get)
        print(f"'{word}': {len(followers)} followers, best=''{best}'' ({followers[best]:.3f})")


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
